## Create Seviri time series - full image 

Step by step full image prediction:
* get matching seviri files in time range
* read and concat seviri files

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import json
import sys
import os
from datetime import datetime, date, timedelta
import xarray as xr
import pandas as pd
import numpy as np
from pyproj import CRS, Transformer
from satpy import find_files_and_readers

In [3]:
import socket
import sys

# get host name to check if we are on euler/daint/iac
hostname = socket.gethostname()
print(hostname)
if hostname.startswith("eu"):
    host = "euler"
else:
    host = "iac"

if host == "euler":
    sys.path.append("/nfs/ch4/wolke_scratch/dnikolo/sat_cirrus")
    sys.path.append("/nfs/ch4/wolke_scratch/dnikolo/sat_cirrus/data")
else:
    sys.path.append("/home/kjeggle/sat-cirrus")
    sys.path.append("/home/kjeggle/sat-cirrus/data")

eu-a2p-417


In [4]:
# todo - these functions are copied from code of the paper amell et al. 2021
from amell_et_al_data import read_seviri_native_file, _resolve_serialisation

In [36]:
def read_and_crop_seviri_data(filepath, crop=None) -> xr.Dataset:
    """reads and crops spatial extent
    
    args:
        filepath (str):
        crop (tuple): (min, max)
    """
    ds = read_seviri_native_file(filepath)
    ds = _resolve_serialisation(ds)
    ds = add_lonlat(ds)
    # print(ds.x.max())
    # print(ds.y.min(),ds.y.max())
    save_coords_path_pre = "/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/original_coords_pre_crop.npy"
    np.save(save_coords_path_pre, {"x": ds.x.values, "y": ds.y.values}, allow_pickle=True)
    if crop:
        lonmin,lonmax,latmin,latmax=crop
        ds = ds.sel(x=slice(lonmax,lonmin),y=slice(latmin,latmax))
    save_coords_path_post = "/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/original_coords_post_crop.npy"
    np.save(save_coords_path_post, {"x": ds.x.values, "y": ds.y.values}, allow_pickle=True)
    
    raise ValueError("Sorry, no numbers below zero")   
    return ds
    

In [49]:
import numpy as np
import xarray as xr
from pathlib import Path

def read_and_crop_seviri_data(filepath, crop=None) -> xr.Dataset:
    """Reads SEVIRI data, optionally crops spatial extent,
    saves x/y coords pre/post crop ONLY ONCE, and validates y shape.

    args:
        filepath (str):
        crop (tuple): (lonmin, lonmax, latmin, latmax)
    """
    ds = read_seviri_native_file(filepath)
    ds = _resolve_serialisation(ds)
    ds = add_lonlat(ds)

    save_coords_path_pre = Path(
        "/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/original_coords_pre_crop.npy"
    )
    save_coords_path_post = Path(
        "/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/original_coords_post_crop.npy"
    )

    # ✅ Save pre-crop coords only if file doesn't exist
    if not save_coords_path_pre.exists():
        np.save(save_coords_path_pre, {"x": ds.x.values, "y": ds.y.values}, allow_pickle=True)

    # ✅ Crop if requested
    if crop:
        lonmin, lonmax, latmin, latmax = crop
        ds = ds.sel(x=slice(lonmax, lonmin), y=slice(latmin, latmax))
        if (ds.lon.values.max()==np.inf) or (ds.lat.values.max()==np.inf):
            raise ValueError("Infinite lon or lat")
        print(ds.lon.values.max(), ds.lon.values.min())
        print(ds.lat.values.max(), ds.lat.values.min())

    # ✅ Save post-crop coords only if file doesn't exist
    if not save_coords_path_post.exists():
        np.save(save_coords_path_post, {"x": ds.x.values, "y": ds.y.values}, allow_pickle=True)

    # ✅ Shape check: y must be exactly (1998,)
    if ds.y.shape != (1998,):
        raise ValueError(f"Unexpected y shape: {ds.y.shape}, expected (1998,)")

    return ds


In [50]:
def add_lonlat(patch: xr.Dataset,
               hor_res: float = 0.01,
               transformer = None):
    """adds lon / lat to patch"""
    if transformer is None:
        # transform lon/lat to cartesian coordinates [m]
        transformer = Transformer.from_crs(seviri_proj, era5_proj, always_xy=True)
        
    patch_lons, patch_lats = transformer.transform(patch.x.values,patch.y.values)
    # print(patch.x.values)
    # print(patch.y.values)
    # patch_lons , _ = transformer.transform(patch.x.values,np.zeros(shape=patch.x.values.shape))
    # _ , patch_lats = transformer.transform(np.zeros(shape=patch.x.values.shape),patch.y.values)
    # print(patch_lons.max(),patch_lons.min())
    # print(patch_lats.max(),patch_lats.min())
    # raise Exceprtion("I got what I needed")
    patch = patch.assign(lon=xr.DataArray(patch_lons, dims="x", coords={'x': patch.x.values}),
                         lat=xr.DataArray(patch_lats, dims="y", coords={'y': patch.y.values}))
    
    patch = patch.assign(latr=lambda x: np.round((np.round(x.lat * (1 / hor_res)) * hor_res).astype('float64'), 4))
    patch = patch.assign(lonr=lambda x: np.round((np.round(x.lon * (1 / hor_res)) * hor_res).astype('float64'), 4))
    
    return patch

In [51]:
from tqdm import tqdm
def create_seviri_timeseries(start_time, 
                             end_time, 
                             domain=(-2.732e+06,2.732e+06,-2.996e+06,2.996e+06),
                             seviri_source_dir="/net/n2o/wolke_scratch2/kjeggle/SEVIRI"):
    """
    Args:
        start_time (datetime)
        end_time (datetime)
        domain (tuple): (lonmin,lonmax,latmin,latmax)
        seviri_source_dir (str)
    
    
    """   
    # get available seviri files
    my_files = find_files_and_readers(base_dir=seviri_source_dir,
                                      reader='seviri_l1b_native',
                                      start_time=start_time,
                                      end_time=end_time)["seviri_l1b_native"]
    my_files = np.sort(my_files)
    print(f"{len(my_files)} SEVIRI files found")
    
    # read seviri files and concat along sensing stop dimension
    ds_list = []
    for fpath in tqdm(my_files):
        # print(fpath)
        ds_list.append(read_and_crop_seviri_data(fpath, crop=domain))
    print("read all seviri files")
    
    # concat files
    ds = xr.concat(ds_list,dim="sensing_stop")
    print("concatenated all files along time dim")
    
    # remove some parameters -> necessary to be able to save as .nc file
    for i in range(1,12):
        try:
            del ds[f"ch{i}"].attrs['orbital_parameters']
        except KeyError:
            pass
        try:
            del ds[f"ch{i}"].attrs['time_parameters']
            #print(f"deleted time_parameters attribute from ch{i}") 
        except KeyError:
            pass
    
    return ds

In [67]:
def create_and_save_seviri_timeseries(date_obj,
                                      domain=(-2.732e+06,2.732e+06,-2.996e+06,2.996e+06),
                                      seviri_source_dir="/net/n2o/wolke_scratch2/kjeggle/SEVIRI",
                                      target_dir="/net/n2o/wolke_scratch2/kjeggle/VerticalCloud/DataSmallDomain/SeviriWholeAreaInput/"):
    print("create seviri timeseries for", date_obj)
    
    start_time = datetime.combine(date_obj, datetime.min.time())
    end_time = start_time + timedelta(hours=23,minutes=59)
    
    ds = create_seviri_timeseries(start_time, end_time, domain, seviri_source_dir)
    
    date_str = datetime.strftime(start_time.date(),"%Y%m%d")
    fname = f"seviri_timeseries_{date_str}.nc"
    print(f"save seviri timeseries ({start_time} - {end_time}) to {fname}")
    ds.to_netcdf(os.path.join(target_dir,fname))
    print("\n")
    print("#############")
    print("\n")

In [68]:
era5_proj = CRS.from_string('EPSG:4326') # latlon
SEVIRI_PROJECTION_SOURCE = "/cluster/work/climate/dnikolo/IceCLoudNet_model/ice_cloud_net/src/data/seviri_proj.json"

with open(SEVIRI_PROJECTION_SOURCE, "r") as json_file:
    data = json.load(json_file)
seviri_proj = CRS.from_json_dict(data)

In [69]:
# create and save dataset for range of dates
start_date = date(2010,1,1)
end_date = date(2010,1,1)
date_list = [start_date + timedelta(days=x) for x in range((end_date-start_date).days+1)]

for d in date_list:
    create_and_save_seviri_timeseries(d, 
                                      seviri_source_dir = "/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/",
                                     target_dir="/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/new_domain_icn",
                                      domain=(-2.732e+06,2.732e+06,-3830000.0,2165000.0))
                                     # domain=(-2.732e+06,2.732e+06,-5504000.0,492000.0))
                                      
                                      # domain=(-2.732e+06,2.732e+06,-2.996e+06,2.996e+06))

create seviri timeseries for 2010-01-01
96 SEVIRI files found


  1%|          | 1/96 [00:04<07:16,  4.59s/it]

29.98145634581617 -29.98145634581617
20.521209224556607 -43.82214717391739


  2%|▏         | 2/96 [00:08<06:30,  4.15s/it]

29.98145634581617 -29.98145634581617
20.521209224556607 -43.82214717391739


...

 99%|█████████▉| 95/96 [10:07<00:07,  7.74s/it]

29.98145634581617 -29.98145634581617
20.521209224556607 -43.82214717391739


100%|██████████| 96/96 [10:15<00:00,  6.41s/it]

29.98145634581617 -29.98145634581617
20.521209224556607 -43.82214717391739
read all seviri files


concatenated all files along time dim
save seviri timeseries (2010-01-01 00:00:00 - 2010-01-01 23:59:00) to seviri_timeseries_20100101.nc


#############




### prep seviri aux file for meta data retrieval

In [90]:
#sev_aux = xr.open_dataset("/net/n2o/wolke_scratch2/kjeggle/VerticalCloud/DataSmallDomain/seviri_aux.nc",decode_times=False)

In [28]:
(5567248-2.996e+06*2)/5567248

-0.0762947869396154

In [ ]:
5500000-2.996e+06*2

In [91]:
# sev_aux = add_lonlat(sev_aux)

# lonmin = -2.732e+06
# lonmax = 2.732e+06
# latmin = -2.996e+06
# latmax = 2.996e+06

# sev_aux = sev_aux.sel(x=slice(lonmax,lonmin),y=slice(latmin,latmax))

In [93]:
# sev_aux.to_netcdf("/net/n2o/wolke_scratch2/kjeggle/VerticalCloud/DataSmallDomain/seviri_aux_latlon.nc")

### manual execution of seviri time series creation

In [23]:
def create_and_save_seviri_timeseries_test(date_obj,
                                      domain=(-2.732e+06,2.732e+06,-2.996e+06,2.996e+06),
                                      seviri_source_dir="/net/n2o/wolke_scratch2/kjeggle/SEVIRI",
                                      target_dir="/net/n2o/wolke_scratch2/kjeggle/VerticalCloud/DataSmallDomain/SeviriWholeAreaInput/"):
    print("create seviri timeseries for", date_obj)
    
    start_time = datetime.combine(date_obj, datetime.min.time())
    end_time = start_time + timedelta(hours=23,minutes=59)
    
    ds = create_seviri_timeseries(start_time, end_time, domain, seviri_source_dir)
    
    date_str = datetime.strftime(start_time.date(),"%Y%m%d")
    fname = f"seviri_timeseries_{date_str}.nc"
    return ds, fname

In [47]:
# create and save dataset for range of dates
start_date = date(2010,1,1)
end_date = date(2010,1,1)
date_list = [start_date + timedelta(days=x) for x in range((end_date-start_date).days+1)]


ds, fname = create_and_save_seviri_timeseries_test(date_list[0], 
                                      seviri_source_dir = "/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/",
                                     target_dir="/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/new_domain_icn",
                                     domain=(-2.732e+06,2.732e+06,-4837500.0,1157359.0))

create seviri timeseries for 2010-01-01
96 SEVIRI files found


100%|██████████| 96/96 [06:23<00:00,  4.00s/it]


read all seviri files
concatenated all files along time dim


In [49]:

def scan_attrs(ds, show_values=False, max_value_len=120):
    def check_mapping(attrs, where):
        for k, v in attrs.items():
            bad = False
            reason = ""

            if isinstance(v, np.ndarray):
                if v.dtype.kind in ("U", "S", "O"):
                    bad = True
                    reason = f"ndarray dtype={v.dtype} shape={v.shape}"
            elif isinstance(v, (list, tuple)):
                # lists of strings often become unicode arrays later
                if any(isinstance(x, str) for x in v):
                    bad = True
                    reason = f"{type(v).__name__} (contains str), len={len(v)}"
            elif isinstance(v, str):
                # plain strings are usually fine; keep off by default
                pass

            if bad:
                print(f"[{where}] attr '{k}': {reason}")
                if show_values:
                    s = str(v)
                    print("   value:", (s[:max_value_len] + "…") if len(s) > max_value_len else s)

    check_mapping(ds.attrs, "GLOBAL")
    for name, var in ds.variables.items():
        check_mapping(var.attrs, f"VAR:{name}")

scan_attrs(ds, show_values=False)

[VAR:ch1] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch2] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch3] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch4] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch5] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch6] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch7] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch8] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch9] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch10] attr 'wavelength': WavelengthRange (contains str), len=4
[VAR:ch11] attr 'wavelength': WavelengthRange (contains str), len=4


In [50]:
for i in range(1,12):
        try:
            del ds[f"ch{i}"].attrs['orbital_parameters']
        except KeyError:
            pass
        try:
            del ds[f"ch{i}"].attrs['time_parameters']
            #print(f"deleted time_parameters attribute from ch{i}") 
        except KeyError:
            pass
        try:
            del ds[f"ch{i}"].attrs['wavelength']
            #print(f"deleted time_parameters attribute from ch{i}") 
        except KeyError:
            pass

In [51]:
ds

<xarray.Dataset> Size: 15GB
Dimensions:        (sensing_stop: 96, y: 1998, x: 1822, sensing_start: 96)
Coordinates:
  * sensing_stop   (sensing_stop) datetime64[ns] 768B 2010-01-01T00:12:42.454...
  * y              (y) float32 8kB -4.835e+06 -4.832e+06 ... 1.154e+06 1.157e+06
  * x              (x) float32 7kB 2.732e+06 2.729e+06 ... -2.729e+06 -2.732e+06
  * sensing_start  (sensing_start) datetime64[ns] 768B 2010-01-01 ... 2010-01...
Data variables: (12/15)
    ch1            (sensing_stop, y, x) float32 1GB nan nan nan ... 0.0 0.0 0.0
    ch2            (sensing_stop, y, x) float32 1GB nan nan nan ... 0.0 0.0 0.0
    ch3            (sensing_stop, y, x) float32 1GB nan nan ... 0.1094 0.1094
    ch4            (sensing_stop, y, x) float32 1GB nan nan nan ... 292.5 292.3
    ch5            (sensing_stop, y, x) float32 1GB nan nan nan ... 239.8 240.1
    ch6            (sensing_stop, y, x) float32 1GB nan nan nan ... 261.7 261.6
    ...             ...
    ch10           (sensing_stop, y, x) float32 1GB nan nan nan ... 293.2 293.2
    ch11           (sensing_stop, y, x) float32 1GB nan nan nan ... 270.6 270.5
    lon            (sensing_stop, x) float64 1MB 29.98 29.93 ... -29.93 -29.98
    lat            (sensing_stop, y) float64 2MB inf inf inf ... 10.56 10.59
    latr           (sensing_stop, y) float64 2MB inf inf inf ... 10.56 10.59
    lonr           (sensing_stop, x) float64 1MB 29.98 29.93 ... -29.93 -29.98
Attributes:
    source_file:                  MSG2-SEVI-MSG15-0100-NA-20100101001242.4540...
    proj4_string:                 +proj=geos +lon_0=0 +h=35785831 +x_0=0 +y_0...
    projection_longitude:         0.0
    projection_latitude:          0.0
    projection_altitude:          35785831.0
    satellite_nominal_longitude:  0.0
    satellite_nominal_latitude:   0.0
    satellite_actual_longitude:   0.026676760545646747
    satellite_actual_latitude:    0.05102085050375031
    satellite_actual_altitude:    35792413.03724569

In [52]:
# target_dir="/nfs/n2o/wolke_scratch/dnikolo/dump/test_time_series/"
target_dir="/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/new_domain_icn/"
# print(f"save seviri timeseries ({start_time} - {end_time}) to {fname}")
ds.to_netcdf(os.path.join(target_dir,fname))
print("\n")
print("#############")
print("\n")



#############




In [53]:
os.path.join(target_dir,fname)

'/nfs/ch4/wolke_scratch/dnikolo/Limited_glaciation/ICN_testing/new_domain_icn/seviri_timeseries_20100101.nc'

In [10]:
seviri_source_dir = "/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI"

In [16]:
# Niamey casestudy
#datetime(2006, 8, 22, 0, 0, 0)-datetime(2006, 8, 22, 23, 59, 0))["

In [11]:
my_files = find_files_and_readers(base_dir=seviri_source_dir,
                                  reader='seviri_l1b_native',
                                  start_time=datetime(2010, 1, 1, 0, 0, 0),
                                  end_time=datetime(2012, 1, 1, 23, 59, 0))["seviri_l1b_native"]

In [12]:
my_files

['/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101092742.300000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101002742.643000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101204241.400000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101132741.813000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101135742.203000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101075742.346000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101215742.350000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-20100101025742.128000000Z-NA.nat',
 '/cluster/work/climate/dnikolo/IceCloudNet_Data/SEVIRI/MSG2-SEVI-MSG15-0100-NA-

In [21]:
# my_files = np.sort([file for file in my_files if not "MSG1-" in file]) # use MSG2 only for those years

In [22]:
96*2

192

In [13]:
my_files = np.sort(my_files)
print(f"{len(my_files)} SEVIRI files found")

96 SEVIRI files found


In [17]:
lonmin = -2.732e+06
lonmax = 2.732e+06
latmin = -2.996e+06
latmax = 2.996e+06

crop_area = (lonmin,lonmax,latmin,latmax)

In [27]:
%%time
# read seviri files and concat along sensing stop dimension
ds_list = [read_and_crop_seviri_data(fpath, crop=crop_area) for fpath in my_files]

<xarray.DataArray 'x' ()> Size: 4B
array(5567248., dtype=float32)
<xarray.DataArray 'y' ()> Size: 4B
array(-5567248., dtype=float32) <xarray.DataArray 'y' ()> Size: 4B
array(5567248., dtype=float32)
<xarray.DataArray 'x' ()> Size: 4B
array(5567248., dtype=float32)
<xarray.DataArray 'y' ()> Size: 4B
array(-5567248., dtype=float32) <xarray.DataArray 'y' ()> Size: 4B
array(5567248., dtype=float32)
<xarray.DataArray 'x' ()> Size: 4B
array(5567248., dtype=float32)
<xarray.DataArray 'y' ()> Size: 4B
array(-5567248., dtype=float32) <xarray.DataArray 'y' ()> Size: 4B
array(5567248., dtype=float32)


KeyboardInterrupt: 

In [ ]:
ds = xr.concat(ds_list,dim="sensing_stop")

In [ ]:
ds

In [38]:
for i in range(1,12):
    try:
        del ds[f"ch{i}"].attrs['orbital_parameters']
    except KeyError:
        pass
    try:
        del ds[f"ch{i}"].attrs['time_parameters']
        #print(f"deleted time_parameters attribute from ch{i}") 
    except KeyError:
        pass

In [39]:
%%time
ds.to_netcdf("/net/n2o/wolke_scratch2/kjeggle/VerticalCloud/DataSmallDomain/SeviriWholeAreaInput/seviri_whole_area_sample_full_day_20240116.nc")

CPU times: user 313 ms, sys: 12 s, total: 12.3 s
Wall time: 28.9 s


In [ ]:
# Niamey station location
niamey_lat= 13.4773
niamey_lon = 2.1758

# niamey_lat_id = 1485

In [ ]:
niamey_x_id = np.argmin(np.abs((ds.isel(sensing_stop=0).lon.values - niamey_lon)))
niamey_y_id = np.argmin(np.abs((ds.isel(sensing_stop=0).lat.values - niamey_lat)))

In [ ]:
print(f"identified lon {ds.isel(sensing_stop=1,x=niamey_x_id).lon.values:.4f} vs actual lon: {niamey_lon} | index in seviri grid: {niamey_x_id}")
print(f"identified lat {ds.isel(sensing_stop=1,y=niamey_y_id).lat.values:.4f} vs actual lat: {niamey_lat} | index in seviri grid: {niamey_y_id}")

In [30]:
km_to_lat = Transformer.from_crs(seviri_proj,era5_proj, always_xy=True)

In [48]:
km_to_lat.transform(-4837500,2.7e6,)

(inf, inf)

In [32]:
lat_to_km = Transformer.from_crs(era5_proj,seviri_proj, always_xy=True)

In [24]:
era5_proj

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [30]:
x,y = lat_to_km.transform(60,60)

In [31]:
x

2438833.4005487235

In [32]:
y

4811749.901981245

In [ ]:
ds.sel(x=x,y=y,method="nearest")

In [45]:
(1+(4811749-5500000)/4811749)*60

51.41786905343566